In [1]:
!pip install chromadb openai pypdf

In [2]:
import chromadb
from openai import OpenAI
from pypdf import PdfReader

client = OpenAI()
chroma_client = chromadb.PersistentClient(path="./chroma_store")
collection = chroma_client.get_or_create_collection("pdf_collection")

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# --- Step 1: Extract text from PDF ---

In [3]:
reader = PdfReader("HR_Policy_Manual_2023.pdf")
pdf_texts = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:  # skip empty pages
        pdf_texts.append({"id": f"page_{i}", "text": text})

# --- Step 2: Add to ChromaDB ---

In [4]:
for d in pdf_texts:
    embedding = get_embedding(d["text"])
    collection.add(
        ids=[d["id"]],
        embeddings=[embedding],
        documents=[d["text"]]
    )

print("PDF content indexed in ChromaDB ✅")

PDF content indexed in ChromaDB ✅


# --- Step 3: Query ---

In [5]:
query_text = "Explain various leave details"
query_embedding = get_embedding(query_text)

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [6]:
print("Query:", query_text)
print("Results:", results)

Query: Explain various leave details
Results: {'ids': [['page_78', 'page_79', 'page_81']], 'embeddings': None, 'documents': [['69\nIIMA HR Policy Manual 2023\n2.2 Leave cannot be claimed as a matter of right. Based on the Institute’s requirement or public \nexigencies, leave can be denied. \n2.3 The leave sanctioning authority may refuse or revoke leaves of any kind but cannot alter the \nkind of leave due and applied for. \n2.4 The reasons for leave should invariably be indicated in the leave application.\n2.5 Any planned leave for more than two days should be applied at least 10 days before the start \nof the leave.\n2.6 On return from a leave of more than ten days, the employee should report for duty to the HoD \nand inform to the HR Office. \n2.7 Leave should be applied through ESS in the same month in which it is availed. \n2.8 Absence without leave not in the continuation of any authorised leave will constitute an \ninterruption of service unless it is regularized.\n(3) EXTENSION